# 12. 이상 후보와 온도 변화를 담은 cal01_base_v2

`1`은 이상 후보, `0`은 이번 기준으로 후보에 넣지 않은 행임.
같은 LOT, 같은 센서에서 **평균 ±3σ를 넘는 값**을 먼저 찾음.
그중 확인한 공통 Z3, Z6 움직임은 센서별로 후보에서 뺌. **같은 행에 다른 센서 후보가 남으면 1을 유지함.**

기록 끝 2시간을 통째로 0으로 바꾸지 않음. v1의 센서값, 파생값, 행 순서와 기존 제외 행을 유지하며, 21개 점수를 모두 계산할 수 없는 행은 라벨을 비움.

TC-Z3~Z6에는 중앙 60%의 중앙값 대비 온도 차이, 5℃ 초과 지속 시간과 누적값을 추가함. 이 값으로 기존 라벨을 다시 바꾸지는 않음.

## 1. 파일 읽기

문자열로 복사해 원본 소수값과 빈칸을 보존하고, 통계 계산에만 별도 숫자 표를 사용함.

In [ ]:
import os
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib import font_manager
from IPython.display import display, Markdown

available_fonts = [font.name for font in font_manager.fontManager.ttflist]
for font_name in ["Malgun Gothic", "AppleGothic", "NanumGothic", "Noto Sans CJK KR"]:
    if font_name in available_fonts:
        plt.rcParams["font.family"] = font_name
        break
plt.rcParams["axes.unicode_minus"] = False

ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)
if not os.path.isdir(os.path.join(ROOT, "notebooks")):
    ROOT = os.path.dirname(ROOT)

PROCESSED_DIR = os.path.join(ROOT, "data", "processed")
INPUT_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v1.csv")
INPUT_METADATA = os.path.join(PROCESSED_DIR, "metadata", "v1", "dataset.json")
OUTPUT_CSV = os.path.join(PROCESSED_DIR, "cal01_base_v2.csv")
REVIEW_DIR = os.path.join(PROCESSED_DIR, "review", "v2")
META_DIR = os.path.join(PROCESSED_DIR, "metadata", "v2")
FIG_DIR = os.path.join(ROOT, "figures", "namelesspark", "12_sigma_labeled_dataframe")
ENC = "utf-8-sig"
SAVE_FILES = True
LABEL_VERSION = "lot_sigma_gt3_common_review_v2_20261020"
SENSORS = [
    "CLN-IN", "TC-Z1", "TC-Z2", "OP-Z1", "OP-Z2",
    "TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6",
    "OP-Z3", "OP-Z4", "OP-Z5", "OP-Z6", "CP-Z4",
    "TC-Z7", "TC-Z8", "TC-Z9", "OP-Z7", "OP-Z8", "TC-OUT1", "TC-OUT2",
]

base = pd.read_csv(INPUT_CSV, encoding=ENC, dtype=str, keep_default_na=False)
output = base.copy()
with open(INPUT_METADATA, encoding="utf-8") as file:
    input_columns = json.load(file)["input_columns"]
GROUPS = {
    'CLN': ['CLN-IN'],
    'Z1': ['TC-Z1', 'OP-Z1'],
    'Z2': ['TC-Z2', 'OP-Z2'],
    'Z3': ['TC-Z3', 'OP-Z3'],
    'Z4': ['TC-Z4', 'OP-Z4', 'CP-Z4'],
    'Z5': ['TC-Z5', 'OP-Z5'],
    'Z6': ['TC-Z6', 'OP-Z6'],
    'Z7': ['TC-Z7', 'OP-Z7'],
    'Z8': ['TC-Z8', 'OP-Z8'],
    'Z9': ['TC-Z9'],
    'COOLING': ['TC-OUT1', 'TC-OUT2'],
}


## 2. LOT별 평균, 표준편차와 점수 계산

기존 제외 행만 빼고 라벨과 관계없이 통계를 계산함. 표준편차는 `ddof=0`.
점수는 `|현재 값−같은 LOT의 평균| / 같은 LOT의 표준편차`. **3 초과**만 범위 밖이며, 3과 같으면 포함하지 않음.

In [ ]:
not_excluded = base["EXCLUDED"].eq("False")
values = base[SENSORS].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan)
reference = values.loc[not_excluded]
groups = base.loc[not_excluded, "LOT_NO"]
lot_mean = reference.groupby(groups).mean()
lot_std = reference.groupby(groups).std(ddof=0)
lot_count = reference.groupby(groups).count()
scores = pd.DataFrame(index=reference.index)
limit_records = []

for sensor in SENSORS:
    mean = groups.map(lot_mean[sensor])
    std = groups.map(lot_std[sensor]).replace(0, np.nan)
    scores[sensor] = reference[sensor].subtract(mean).divide(std).abs()
    for lot in lot_mean.index:
        center, spread = lot_mean.loc[lot, sensor], lot_std.loc[lot, sensor]
        limit_records.append({
            "LOT_NO": lot, "SENSOR": sensor, "COUNT": int(lot_count.loc[lot, sensor]),
            "MEAN": center, "STD_DDOF0": spread, "LOWER": center - 3 * spread, "UPPER": center + 3 * spread,
        })

limits = pd.DataFrame(limit_records)
valid = pd.Series(False, index=base.index)
valid.loc[reference.index] = np.isfinite(scores.to_numpy()).all(axis=1)
outside = scores.gt(3).any(axis=1)
times = pd.to_datetime(base["MEAS_DT"])


## 3. 확인한 공통 움직임만 후보에서 빼기

Z3 온도가 올랐다 내려간 뒤 Z6가 늦게 낮아졌다 회복하는 모양을 LOT끼리 비교했음.
기록 끝 약 93~101분 전 Z3 고점을 중심으로 확인한 **13개 LOT의 한 회차씩** 적용함.
`TC-Z3`, `OP-Z3`, `TC-Z6`, `OP-Z6` 중 아래 표에 적은 실제 초과 구간만 후보에서 뺌.
Z4, Z5와 CP 등 나머지 센서는 그대로 둠. 같은 회차 안에서도 따로 튄 값은 남김.

표의 시작과 끝은 **후보에서 뺄 관측값의 범위**임. 공정 시작, 종료 시각을 뜻하지 않으며 끝은 포함하지 않음.
133은 앞부분 관측이 짧아 보이는 값만 판단함. 044는 관측이 짧아 이 표에 넣지 않음.
검토한 CSV와 내용이 달라지면 이 표를 자동 적용하지 않음.

In [ ]:
REVIEWED_SOURCE_SHA256 = "b384f4ba7a985142f573042d13bc30a3dac7038eb1fdef9eed18329b8fe7002a"
with open(INPUT_CSV, "rb") as file:
    current_source_sha256 = hashlib.file_digest(file, "sha256").hexdigest()
review_applied = current_source_sha256 == REVIEWED_SOURCE_SHA256

common_ranges = pd.DataFrame([
    ['240001', 'TC-Z3', '2024-01-02 13:26:17', '2024-01-02 13:33:32', 'Z3 온도가 크게 올라간 봉우리'],
    ['240001', 'TC-Z3', '2024-01-02 13:36:45', '2024-01-02 13:40:40', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240001', 'OP-Z3', '2024-01-02 13:25:15', '2024-01-02 13:33:11', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240001', 'TC-Z6', '2024-01-02 13:35:23', '2024-01-02 13:40:04', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240001', 'TC-Z6', '2024-01-02 13:45:53', '2024-01-02 13:51:58', 'Z6 온도가 다시 올라간 부분'],
    ['240001', 'OP-Z6', '2024-01-02 13:33:33', '2024-01-02 13:33:51', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240001', 'OP-Z6', '2024-01-02 13:37:11', '2024-01-02 13:42:12', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240001', 'OP-Z6', '2024-01-02 13:47:30', '2024-01-02 13:51:52', 'Z6 온도가 다시 올라갈 때 출력이 낮아진 부분'],
    ['240024', 'TC-Z3', '2024-03-08 04:51:02', '2024-03-08 04:58:33', 'Z3 온도가 크게 올라간 봉우리'],
    ['240024', 'TC-Z3', '2024-03-08 05:06:56', '2024-03-08 05:12:00', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240024', 'OP-Z3', '2024-03-08 04:50:13', '2024-03-08 04:59:04', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240024', 'TC-Z6', '2024-03-08 05:00:36', '2024-03-08 05:07:40', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240024', 'TC-Z6', '2024-03-08 05:13:54', '2024-03-08 05:22:47', 'Z6 온도가 다시 올라간 부분'],
    ['240024', 'OP-Z6', '2024-03-08 04:59:07', '2024-03-08 05:12:02', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240037', 'TC-Z3', '2024-04-14 07:51:15', '2024-04-14 07:58:22', 'Z3 온도가 크게 올라간 봉우리'],
    ['240037', 'TC-Z3', '2024-04-14 08:02:53', '2024-04-14 08:06:37', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240037', 'OP-Z3', '2024-04-14 07:50:21', '2024-04-14 07:58:12', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240037', 'TC-Z6', '2024-04-14 08:00:27', '2024-04-14 08:07:41', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240037', 'TC-Z6', '2024-04-14 08:11:08', '2024-04-14 08:17:53', 'Z6 온도가 다시 올라간 부분'],
    ['240037', 'OP-Z6', '2024-04-14 08:00:22', '2024-04-14 08:08:56', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240038', 'TC-Z3', '2024-04-16 12:06:47', '2024-04-16 12:08:07', 'Z3 온도가 크게 올라간 봉우리'],
    ['240038', 'TC-Z3', '2024-04-16 12:16:04', '2024-04-16 12:19:08', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240038', 'OP-Z3', '2024-04-16 12:04:01', '2024-04-16 12:10:40', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240038', 'TC-Z6', '2024-04-16 12:24:50', '2024-04-16 12:28:31', 'Z6 온도가 다시 올라간 부분'],
    ['240038', 'OP-Z6', '2024-04-16 12:16:49', '2024-04-16 12:19:22', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240058', 'TC-Z3', '2024-06-09 07:26:03', '2024-06-09 07:32:03', 'Z3 온도가 크게 올라간 봉우리'],
    ['240058', 'OP-Z3', '2024-06-09 07:24:57', '2024-06-09 07:31:37', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240058', 'TC-Z6', '2024-06-09 07:37:05', '2024-06-09 07:40:38', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240058', 'OP-Z6', '2024-06-09 07:36:19', '2024-06-09 07:44:52', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240061', 'TC-Z3', '2024-06-15 03:14:01', '2024-06-15 03:18:01', 'Z3 온도가 크게 올라간 봉우리'],
    ['240061', 'TC-Z6', '2024-06-15 03:26:03', '2024-06-15 03:26:13', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240061', 'OP-Z6', '2024-06-15 03:24:16', '2024-06-15 03:28:48', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240069', 'TC-Z3', '2024-07-08 04:51:27', '2024-07-08 04:57:43', 'Z3 온도가 크게 올라간 봉우리'],
    ['240069', 'OP-Z3', '2024-07-08 04:50:27', '2024-07-08 04:57:04', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240069', 'TC-Z6', '2024-07-08 05:03:14', '2024-07-08 05:07:14', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240069', 'TC-Z6', '2024-07-08 05:10:41', '2024-07-08 05:17:22', 'Z6 온도가 다시 올라간 부분'],
    ['240069', 'OP-Z6', '2024-07-08 04:59:56', '2024-07-08 05:09:12', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240091', 'TC-Z3', '2024-09-09 07:07:43', '2024-09-09 07:15:04', 'Z3 온도가 크게 올라간 봉우리'],
    ['240091', 'TC-Z3', '2024-09-09 07:23:56', '2024-09-09 07:29:22', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240091', 'OP-Z3', '2024-09-09 07:06:43', '2024-09-09 07:15:17', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240091', 'TC-Z6', '2024-09-09 07:17:22', '2024-09-09 07:22:58', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240091', 'TC-Z6', '2024-09-09 07:30:32', '2024-09-09 07:39:25', 'Z6 온도가 다시 올라간 부분'],
    ['240091', 'OP-Z6', '2024-09-09 07:14:54', '2024-09-09 07:26:08', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240091', 'OP-Z6', '2024-09-09 07:31:15', '2024-09-09 07:39:52', 'Z6 온도가 다시 올라갈 때 출력이 낮아진 부분'],
    ['240108', 'TC-Z3', '2024-10-18 10:14:29', '2024-10-18 10:17:41', 'Z3 온도가 크게 올라간 봉우리'],
    ['240108', 'TC-Z3', '2024-10-18 10:24:39', '2024-10-18 10:27:22', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240108', 'OP-Z3', '2024-10-18 10:12:36', '2024-10-18 10:18:21', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240108', 'TC-Z6', '2024-10-18 10:22:28', '2024-10-18 10:27:25', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240108', 'TC-Z6', '2024-10-18 10:31:59', '2024-10-18 10:37:52', 'Z6 온도가 다시 올라간 부분'],
    ['240108', 'OP-Z6', '2024-10-18 10:21:08', '2024-10-18 10:24:24', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240108', 'OP-Z6', '2024-10-18 10:24:25', '2024-10-18 10:28:52', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240114', 'TC-Z3', '2024-11-04 19:15:40', '2024-11-04 19:22:12', 'Z3 온도가 크게 올라간 봉우리'],
    ['240114', 'TC-Z3', '2024-11-04 19:27:28', '2024-11-04 19:30:03', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240114', 'OP-Z3', '2024-11-04 19:14:08', '2024-11-04 19:22:40', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240114', 'TC-Z6', '2024-11-04 19:26:25', '2024-11-04 19:30:42', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240114', 'TC-Z6', '2024-11-04 19:34:28', '2024-11-04 19:41:02', 'Z6 온도가 다시 올라간 부분'],
    ['240114', 'OP-Z6', '2024-11-04 19:23:48', '2024-11-04 19:32:19', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240114', 'OP-Z6', '2024-11-04 19:32:24', '2024-11-04 19:32:32', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240117', 'TC-Z3', '2024-11-13 12:56:18', '2024-11-13 13:02:20', 'Z3 온도가 크게 올라간 봉우리'],
    ['240117', 'OP-Z3', '2024-11-13 12:55:18', '2024-11-13 13:01:48', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240117', 'TC-Z6', '2024-11-13 13:07:56', '2024-11-13 13:10:48', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240117', 'TC-Z6', '2024-11-13 13:19:26', '2024-11-13 13:26:41', 'Z6 온도가 다시 올라간 부분'],
    ['240117', 'OP-Z6', '2024-11-13 13:04:57', '2024-11-13 13:10:14', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240117', 'OP-Z6', '2024-11-13 13:10:15', '2024-11-13 13:13:25', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240121', 'TC-Z3', '2024-11-24 16:39:09', '2024-11-24 16:46:55', 'Z3 온도가 크게 올라간 봉우리'],
    ['240121', 'TC-Z3', '2024-11-24 16:51:56', '2024-11-24 16:56:59', 'Z3 온도가 봉우리 뒤 낮아진 부분'],
    ['240121', 'OP-Z3', '2024-11-24 16:38:07', '2024-11-24 16:46:55', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240121', 'TC-Z6', '2024-11-24 16:48:36', '2024-11-24 16:55:42', 'Z6 온도가 뒤늦게 낮아진 부분'],
    ['240121', 'TC-Z6', '2024-11-24 16:59:16', '2024-11-24 17:07:13', 'Z6 온도가 다시 올라간 부분'],
    ['240121', 'OP-Z6', '2024-11-24 16:49:30', '2024-11-24 16:58:00', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
    ['240133', 'TC-Z3', '2024-12-28 15:08:43', '2024-12-28 15:11:51', 'Z3 온도가 크게 올라간 봉우리'],
    ['240133', 'OP-Z3', '2024-12-28 15:06:24', '2024-12-28 15:13:15', 'Z3 온도가 올라갈 때 출력이 낮아진 부분'],
    ['240133', 'TC-Z6', '2024-12-28 15:28:37', '2024-12-28 15:31:53', 'Z6 온도가 다시 올라간 부분'],
    ['240133', 'OP-Z6', '2024-12-28 15:16:31', '2024-12-28 15:22:41', 'Z6 온도가 낮아질 때 출력이 올라간 부분'],
], columns=["LOT_NO", "SENSOR", "START", "END", "REASON"])
common_ranges["START"] = pd.to_datetime(common_ranges["START"])
common_ranges["END"] = pd.to_datetime(common_ranges["END"])
common_ranges.insert(0, "REVIEW_ID", ["C" + str(i).zfill(3) for i in range(1, len(common_ranges) + 1)])

reviewed_events = pd.DataFrame([
    ['240001', '2024-01-02 13:29:00'],
    ['240024', '2024-03-08 04:53:00'],
    ['240037', '2024-04-14 07:53:00'],
    ['240038', '2024-04-16 12:07:00'],
    ['240058', '2024-06-09 07:28:00'],
    ['240061', '2024-06-15 03:15:00'],
    ['240069', '2024-07-08 04:53:00'],
    ['240091', '2024-09-09 07:10:00'],
    ['240108', '2024-10-18 10:15:00'],
    ['240114', '2024-11-04 19:18:00'],
    ['240117', '2024-11-13 12:58:00'],
    ['240121', '2024-11-24 16:42:00'],
    ['240133', '2024-12-28 15:10:00'],
], columns=["LOT_NO", "PEAK"])
reviewed_events["PEAK"] = pd.to_datetime(reviewed_events["PEAK"])

stat_flags = scores.gt(3).reindex(base.index, fill_value=False)
stat_flags = stat_flags.mul(valid, axis=0).astype(bool)
removed_flags = pd.DataFrame(False, index=base.index, columns=SENSORS)
removed_review_ids = pd.Series("", index=base.index)
range_counts = []

for row in common_ranges.itertuples(index=False):
    selected = base["LOT_NO"].eq(row.LOT_NO) & times.ge(row.START) & times.lt(row.END)
    selected = selected & stat_flags[row.SENSOR] & review_applied
    removed_flags.loc[selected, row.SENSOR] = True
    removed_review_ids.loc[selected] += row.REVIEW_ID + ";"
    range_counts.append(int(selected.sum()))

common_ranges["REMOVED_SENSOR_ROWS"] = range_counts
candidate_flags = stat_flags & ~removed_flags
common_rows = removed_flags.any(axis=1)
outside = candidate_flags.any(axis=1)
print("검토표 적용" if review_applied else "CSV가 달라 검토표 미적용: ±3σ 통계 후보만 계산함.")
display(common_ranges.rename(columns={
    "REVIEW_ID": "번호", "LOT_NO": "LOT", "SENSOR": "센서",
    "START": "시작", "END": "끝(미포함)", "REASON": "공통 움직임", "REMOVED_SENSOR_ROWS": "후보에서 뺀 개수",
}))


## 4. 남은 후보로 라벨 붙이기

공통 움직임을 뺀 뒤 **21개 센서 중 후보가 하나라도 남으면 1**, 없으면 0.
`EVIDENCE_SENSOR`에는 남은 센서만 적음. 공통 움직임으로 뺀 센서와 해당 행은 별도 표에 보관함.
기존 문맥 표시와 후보 연결은 v1의 분석 정보로 남기며 새 라벨 판정이나 모델 입력에 사용하지 않음.

In [ ]:
old_evidence = [
    "EVIDENCE_WO", "EVIDENCE_MNT_TYPE", "EVIDENCE_SENSOR", "EVIDENCE_CANDIDATE_IDS", "ASSOCIATION_KIND",
    "EVIDENCE_DOWN_MIN", "EVIDENCE_HAS_PART_RECORD", "EVIDENCE_PART_NAMES",
    "EVIDENCE_PART_QTY_TOTAL", "EVIDENCE_PART_CHANGE_REASONS", "EVIDENCE_FAIL_MODE",
]
output.loc[not_excluded, old_evidence] = ""
output.loc[not_excluded, "LABEL"] = ""
output.loc[not_excluded, "LABEL_VERSION"] = LABEL_VERSION
output.loc[not_excluded, "LABEL_STATUS"] = "21개 센서 점수 계산 불가"
output.loc[not_excluded, "LABEL_REASON"] = "센서 결측 또는 표준편차 0 등으로 21개 점수를 모두 계산할 수 없어 라벨을 비움."
output.loc[not_excluded, "RULE"] = "lot_sigma_unavailable"
output.loc[not_excluded, "EVIDENCE_CONFIDENCE"] = "점수 계산 불가"
output.loc[valid, "LABEL"] = outside.loc[valid].astype(int).astype(str)
output.loc[valid, "RULE"] = "any_21_sensor_abs_z_gt_3"
output.loc[valid, "LABEL_STATUS"] = "LOT별 통계 후보와 공통 움직임 구분" if review_applied else "LOT별 통계 후보, 공통 검토표 미적용"
output.loc[valid, "EVIDENCE_CONFIDENCE"] = "같은 LOT 전체 관측으로 계산한 통계 기준"
output.loc[valid & output["LABEL"].eq("0"), "LABEL_REASON"] = "21개 센서 모두 같은 LOT의 평균±3σ 범위 안 또는 경계에 있어 후보로 선정하지 않음."
output.loc[valid & output["LABEL"].eq("1"), "LABEL_REASON"] = "21개 센서 중 하나 이상이 같은 LOT의 평균±3σ 범위 밖이라 이상 후보로 선정함."

for sensor in SENSORS:
    hit = base.index[candidate_flags[sensor]]
    output.loc[hit, "EVIDENCE_SENSOR"] += sensor + ";"
output.loc[not_excluded, "EVIDENCE_SENSOR"] = output.loc[not_excluded, "EVIDENCE_SENSOR"].str.rstrip(";")
output["MODEL_ELIGIBLE"] = (valid & base["FEATURES_COMPLETE"].eq("True")).map({True: "True", False: "False"})
output.loc[common_rows, "RULE"] = "lot_sigma_gt3_reviewed_common_removed"
output.loc[common_rows, "EVIDENCE_CONFIDENCE"] = "같은 LOT의 통계 기준과 공통 움직임 검토표"
output.loc[common_rows & output["LABEL"].eq("0"), "LABEL_REASON"] = "3σ 밖인 값이 검토한 공통 움직임에만 해당해 후보에서 뺌."
output.loc[common_rows & output["LABEL"].eq("1"), "LABEL_REASON"] = "공통 움직임은 후보에서 뺐지만 다른 센서의 3σ 초과 후보가 남아 1을 유지함."

common_audit = output.loc[common_rows, ["ROW_ID", "LOT_NO", "MEAS_DT", "LABEL", "EVIDENCE_SENSOR"]].copy()
common_audit["STAT_LABEL"] = stat_flags.loc[common_rows].any(axis=1).astype(int)
common_audit["REMOVED_SENSORS"] = ""
common_audit["REVIEW_IDS"] = removed_review_ids.loc[common_rows].str.rstrip(";")
for sensor in SENSORS:
    hit = common_audit.index[removed_flags.loc[common_audit.index, sensor]]
    common_audit.loc[hit, "REMOVED_SENSORS"] += sensor + ";"
common_audit["REMOVED_SENSORS"] = common_audit["REMOVED_SENSORS"].str.rstrip(";")


## 5. 라벨 개수와 변경 내용

통계 후보를 만든 뒤 공통 움직임만 뺀 전후를 비교함.
같은 Z-score로 만든 라벨을 같은 Z-score로 다시 채점하는 것은 독립적인 성능 평가가 아님.

### 통계, 도메인, 분석 라벨 읽기

- `STAT_LABEL`: 공통 움직임을 해제하기 전의 `평균 ±3σ` 초과 여부.
  - 같은 LOT의 제외하지 않은 기록으로 계산함. 고장 정답은 아님.
- `DOMAIN_LABEL`: 검토해서 후보를 해제하면 `0`, 근거를 확인해 후보로 채택하면 `1`.
  - 검토하지 않은 곳은 빈칸. 기존 규칙이 만든 `1`을 검토 완료로 복사하지 않음.
- `LABEL`: 해당 버전에서 분석에 사용하는 기존 라벨.
  - `0`도 정상 확정이라는 뜻은 아님. 제외와 보류는 빈칸으로 유지함.
- 구역은 `Z4__STAT_LABEL`, `Z4__DOMAIN_LABEL`, `Z4__LABEL`처럼 표시함.
  - 세 라벨과 판단 이유는 모델 입력 `X`에 넣지 않음.


In [ ]:
stat_labels = pd.Series("", index=base.index)
stat_labels.loc[valid] = stat_flags.loc[valid].any(axis=1).astype(int).astype(str)
label_counts = pd.DataFrame({
    "v1": base["LABEL"].replace("", "라벨 없음").value_counts(),
    "공통 움직임 구분 전": stat_labels.replace("", "라벨 없음").value_counts(),
    "v2": output["LABEL"].replace("", "라벨 없음").value_counts(),
}).fillna(0).astype(int)
lot_label_counts = pd.crosstab(output["LOT_NO"], output["LABEL"].replace("", "라벨 없음"))
display(label_counts)
display(lot_label_counts)

changed = base["LABEL"].ne(output["LABEL"])
changes = output.loc[changed, ["ROW_ID", "LOT_NO", "MEAS_DT"]].copy()
for column in ["LABEL", "LABEL_REASON", "EVIDENCE_SENSOR", "EVIDENCE_WO", "EVIDENCE_CANDIDATE_IDS"]:
    changes["OLD_" + column] = base.loc[changed, column]
    changes["NEW_" + column] = output.loc[changed, column]
print("라벨이 바뀐 행:", int(changed.sum()), "/ 새 CSV:", OUTPUT_CSV)
common_lot_counts = pd.crosstab(common_audit["LOT_NO"], common_audit["LABEL"]).reindex(columns=["0", "1"], fill_value=0)
common_lot_counts.columns = ["최종0", "다른후보가 남아 최종1"]
common_lot_counts.insert(0, "공통움직임을 뺀 행", common_audit.groupby("LOT_NO").size())
display(common_lot_counts)

# v1에서 물려받은 통계/구역 라벨을 v2 계산 결과로 갱신함.
output["DATA_KIND"] = "raw"
output["DATASET_VERSION"] = "v2"
output["STAT_LABEL"] = stat_labels
output["DOMAIN_LABEL"] = ""
cleared = valid & stat_labels.eq("1") & output["LABEL"].eq("0") & common_rows
output.loc[cleared, "DOMAIN_LABEL"] = "0"

for group, group_sensors in GROUPS.items():
    available = scores[group_sensors].notna().all(axis=1) & not_excluded
    group_flags = scores[group_sensors].gt(3).reindex(base.index, fill_value=False)
    before = group_flags.any(axis=1)
    remaining = (group_flags & ~removed_flags[group_sensors]).any(axis=1)
    cleared = available & before & ~remaining & removed_flags[group_sensors].any(axis=1)
    output[group + "__STAT_LABEL"] = before.astype("Int64").where(available)
    output[group + "__DOMAIN_LABEL"] = pd.Series(pd.NA, index=base.index, dtype="Int64")
    output.loc[cleared, group + "__DOMAIN_LABEL"] = 0
    output[group + "__LABEL"] = remaining.astype("Int64").where(available)
    output[group + "__LABEL_STATUS"] = "같은 LOT 통계 후보와 공통 움직임 검토"
    output.loc[~available, group + "__LABEL_STATUS"] = "제외 또는 점수 계산 불가"
    output[group + "__LABEL_REASON"] = "같은 구역의 3σ 초과 후보에서 검토한 공통 움직임만 뺌."
    output[group + "__LABEL_SENSORS"] = ""
    for sensor in group_sensors:
        hit = available & group_flags[sensor] & ~removed_flags[sensor]
        output.loc[hit, group + "__LABEL_SENSORS"] += sensor + ";"
    output[group + "__LABEL_SENSORS"] = output[group + "__LABEL_SENSORS"].str.rstrip(";")
    output[group + "__COMMON_REVIEW_IDS"] = removed_review_ids.where(removed_flags[group_sensors].any(axis=1), "").str.rstrip(";")


## 6. 회색 점과 주황 점을 함께 보기

각 표 바로 아래에 같은 LOT의 Z3, Z6와 CP를 그렸음.
**회색 점은 공통 움직임으로 후보에서 뺀 값**, **주황 점은 남긴 후보**임.
노란 배경은 후보에서 뺀 관측값의 시간 범위이며, 회색 점과 같은 위치임.
가로 점선은 해당 LOT의 평균 ±3σ. 날짜와 시각은 모두 2024년 실제 기록임.

In [ ]:
common_plot_sensors = ["TC-Z3", "OP-Z3", "TC-Z6", "OP-Z6", "CP-Z4"]
if SAVE_FILES:
    os.makedirs(FIG_DIR, exist_ok=True)

for event in reviewed_events.itertuples(index=False):
    lot, peak = event.LOT_NO, event.PEAK
    view = base["LOT_NO"].eq(lot) & times.ge(peak - pd.Timedelta(minutes=25)) & times.le(peak + pd.Timedelta(minutes=35))
    index = base.index[view]
    table = pd.DataFrame({
        "센서": common_plot_sensors,
        "3σ 초과": stat_flags.loc[index, common_plot_sensors].sum().to_numpy(),
        "공통 움직임으로 뺌": removed_flags.loc[index, common_plot_sensors].sum().to_numpy(),
        "남긴 센서 후보": candidate_flags.loc[index, common_plot_sensors].sum().to_numpy(),
    })
    display(Markdown("### LOT " + lot + " | Z3 고점 " + str(peak)))
    display(table)
    figure, axes = plt.subplots(5, 1, figsize=(13, 12), sharex=True, layout="constrained")
    for axis, channel in zip(axes, common_plot_sensors):
        axis.plot(times.loc[index], values.loc[index, channel], color="#355f82", linewidth=0.9)
        gone = index[removed_flags.loc[index, channel]]
        kept = index[candidate_flags.loc[index, channel]]
        for row in common_ranges.loc[common_ranges["LOT_NO"].eq(lot) & common_ranges["SENSOR"].eq(channel)].itertuples(index=False):
            if review_applied:
                axis.axvspan(row.START, row.END, color="#f9db73", alpha=0.3, label="노란 배경: 뺀 값의 시간 범위")
        axis.scatter(times.loc[gone], values.loc[gone, channel], s=9, color="#83878d", zorder=3, label="회색: 공통 움직임으로 뺌")
        axis.scatter(times.loc[kept], values.loc[kept, channel], s=10, color="#df650b", zorder=4, label="주황: 남긴 후보")
        mean, std = lot_mean.loc[lot, channel], lot_std.loc[lot, channel]
        axis.axhline(mean - 3 * std, color="#747d87", linestyle="--", linewidth=0.8, label="점선: LOT 평균 ±3σ")
        axis.axhline(mean + 3 * std, color="#747d87", linestyle="--", linewidth=0.8)
        axis.set_ylabel(channel + ("\n(단위 미확인)" if channel.startswith("CP") else (" (%)" if channel.startswith("OP") else " (℃)")))
        axis.grid(alpha=0.15)
    handles, labels = axes[0].get_legend_handles_labels()
    legend_items = dict(zip(labels, handles))
    axes[0].legend(legend_items.values(), legend_items.keys(), loc="lower left", bbox_to_anchor=(0, 1.02), fontsize=9, ncol=2)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[-1].set_xlabel("2024년 실제 날짜와 시각")
    figure.suptitle(lot + " | 회색은 공통 움직임, 주황은 남긴 후보", fontsize=14)
    if SAVE_FILES:
        figure.savefig(os.path.join(FIG_DIR, "common_" + lot + ".png"), dpi=130)
    plt.show()
    plt.close(figure)
    if lot == "240133":
        display(Markdown("133은 고점 앞 관측이 13분뿐임. 기록되지 않은 앞부분은 추정하지 않음."))


## 7. 별도로 남긴 후보와 0으로 바뀐 값 읽기

노란 배경은 표의 시간 범위임. **노란 배경 전체를 1로 붙이지 않음.**
주황 점은 해당 센서의 남긴 후보, 회색 점은 공통 움직임으로 뺀 값임. 가로 점선은 평균 ±3σ.
표에는 해당 센서 후보와 다른 센서까지 포함한 최종 `LABEL` 개수를 함께 표시함.
시간은 2024년 실제 날짜와 시각이며 시작은 포함, 끝은 제외함.

In [ ]:
review_cases = [
    {
        "lot": "240024", "sensor": "CP-Z4",
        "start": "2024-03-08 04:55", "end": "2024-03-08 05:03",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "CP가 0.45 부근에서 0.383640까지 내려갔다 회복함. 같은 강종 108의 최저 0.404613보다 낮아 다시 살펴볼 이유가 있음.",
        "record": "03/10 15:10 정기점검 이상없음 감지 기록이 있음(WO-2024-00378). 이 기록을 라벨 조건으로 쓰지는 않음.",
    },
    {
        "lot": "240037", "sensor": "CP-Z4",
        "start": "2024-04-13 23:57", "end": "2024-04-14 00:00",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "온도 변화는 작은데 CP는 0.606760까지 올라감. 같은 강종 121, 133의 최고 0.466862보다 큰 급등임.",
        "record": "강종 변경 41분 뒤임. 04/14 15:04 확인 중 감지 기록과 실제 작업일 04/23은 구분함(WO-2024-00627).",
    },
    {
        "lot": "240037", "sensor": "CP-Z4",
        "start": "2024-04-14 03:10", "end": "2024-04-14 03:12",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "CP가 0.522137까지 짧게 다시 올라감. 앞의 23:57 급등과 약 3시간 떨어진 별도 변화임.",
        "record": "같은 날 15:04 확인 중 감지 기록이 있음(WO-2024-00627). 두 급등 사이를 이어서 1로 붙이지 않음.",
    },
    {
        "lot": "240069", "sensor": "CP-Z4",
        "start": "2024-07-08 02:30", "end": "2024-07-08 02:40",
        "plot": ["CP-Z4", "TC-Z4", "OP-Z4"],
        "reason": "CP가 약 0.45에서 0.909111까지 크게 올라갔다 복귀함. 1분 평균 최고도 0.883902로, 같은 강종 044, 061, 091의 최고 0.478683보다 큼.",
        "record": "08:49 절연 저하 감지, 11:11 작업 시작, O링 3개 예방 교체 기록이 있음(WO-2024-01094). CP 급등과의 연결은 별도로 살펴볼 내용임.",
    },
    {
        "lot": "240108", "sensor": "TC-Z4",
        "start": "2024-10-18 05:50", "end": "2024-10-18 06:20",
        "plot": ["TC-Z4", "OP-Z4"],
        "reason": "1분 평균으로 본 30분 온도 변화 폭은 2.70℃임. 같은 강종 024의 최대 30분 변화 폭 1.42℃보다 큼. 온도와 출력이 반대로 움직이는 사실 자체를 이상 근거로 삼지는 않음.",
        "record": "15:50 원인 미상 감지 기록이 있음. 실제 작업은 10/26 이후임(WO-2024-01678).",
    },
    {
        "lot": "240117", "sensor": "CP-Z4",
        "start": "2024-11-13 13:02", "end": "2024-11-13 13:10",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "CP가 약 0.45에서 0.367097까지 내려갔다 회복함. 09:00 이전 승온 제외 구간과는 다른 변화임.",
        "record": "마모량 초과 작업은 12:44에 끝남(WO-2024-01854). 큰 하강은 13:02:50부터라 종료 18분 50초 뒤임. 같은 강종 비교 LOT는 없음.",
    },
    {
        "lot": "240121", "sensor": "CP-Z4",
        "start": "2024-11-24 14:26:05", "end": "2024-11-24 14:27:09",
        "plot": ["CP-Z4", "TC-Z4"],
        "reason": "CP가 0.45 부근에서 0.422714까지 짧게 내려감. 아래 16시대 Z6 반등과 시간과 센서가 다르므로 별도로 판단함.",
        "record": "다음 날 14:19 설비 트립 감지, 11/30 작업 기록이 있음(WO-2024-01922). 라벨은 이 시간 연관이 아니라 ±3σ로 계산함.",
    },
    {
        "lot": "240121", "sensor": "TC-Z6",
        "start": "2024-11-24 16:56:02", "end": "2024-11-24 17:01:47",
        "plot": ["TC-Z6", "OP-Z6", "CP-Z4"],
        "reason": "Z3 고점 16:42, Z6 저점 16:53 뒤의 반등임. 037, 133에도 비슷한 반등이 있어 검토한 TC-Z6 초과값은 공통 움직임으로 후보에서 뺌.",
        "record": "같은 시간에 CP 등 다른 센서 후보가 남은 행은 1을 유지함. 이 시간 범위 전체를 0으로 바꾸지는 않음.",
    },
]
times = pd.to_datetime(base["MEAS_DT"])
window_records = []
window_row_tables = []

if SAVE_FILES:
    os.makedirs(FIG_DIR, exist_ok=True)


In [ ]:
for number, case in enumerate(review_cases, 1):
    lot, sensor = case["lot"], case["sensor"]
    start, end = pd.Timestamp(case["start"]), pd.Timestamp(case["end"])
    selected = base["LOT_NO"].eq(lot) & times.ge(start) & times.lt(end)
    sensor_hit = candidate_flags.loc[selected, sensor]
    sensor_removed = removed_flags.loc[selected, sensor]
    labels = output.loc[selected, "LABEL"]
    record = {
        "LOT": lot, "센서": sensor, "시작": start, "끝(미포함)": end,
        "이 센서로 후보 1": int(sensor_hit.sum()),
        "이 센서의 공통 움직임으로 뺌": int(sensor_removed.sum()),
        "전체 센서 기준 LABEL 1": int(labels.eq("1").sum()),
        "전체 센서 기준 LABEL 0": int(labels.eq("0").sum()),
        "라벨 없음": int(labels.eq("").sum()),
    }
    display(Markdown("### " + str(number) + ". " + lot + " " + sensor))
    display(pd.DataFrame([record]))

    view = base["LOT_NO"].eq(lot) & times.ge(start - pd.Timedelta(minutes=5)) & times.lt(end + pd.Timedelta(minutes=5))
    index = base.index[view]
    figure, axes = plt.subplots(len(case["plot"]), 1, figsize=(13, 2.7 * len(case["plot"])), sharex=True, layout="constrained")
    for axis, channel in zip(axes, case["plot"]):
        mean, std = lot_mean.loc[lot, channel], lot_std.loc[lot, channel]
        axis.axvspan(start, end, color="#f9db73", alpha=0.28, label="노란 배경: 표의 시간 범위")
        axis.plot(times.loc[index], values.loc[index, channel], color="#355f82", linewidth=1, label="센서 원자료")
        hit = candidate_flags.loc[index, channel]
        gone = removed_flags.loc[index, channel]
        axis.scatter(times.loc[index[gone]], values.loc[index[gone], channel], s=9, color="#83878d", zorder=3, label="회색점: 공통 움직임으로 뺌")
        axis.scatter(times.loc[index[hit]], values.loc[index[hit], channel], s=9, color="#df650b", zorder=3, label="주황점: 남긴 후보")
        axis.axhline(mean - 3 * std, color="#747d87", linestyle="--", linewidth=1, label="평균 ±3σ")
        axis.axhline(mean + 3 * std, color="#747d87", linestyle="--", linewidth=1)
        axis.set_ylabel(channel + ("\n(단위 미확인)" if channel.startswith("CP") else (" (%)" if channel.startswith("OP") else " (℃)")))
        axis.grid(alpha=0.15)
    axes[0].legend(loc="lower left", bbox_to_anchor=(0, 1.02), fontsize=8, ncol=3)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[-1].set_xlabel("2024년 실제 날짜와 시각")
    figure.suptitle(lot + " | " + case["start"] + " ~ " + case["end"], fontsize=13)
    if SAVE_FILES:
        filename = f"{number:02d}_{lot}_{sensor}_sigma3.png"
        figure.savefig(os.path.join(FIG_DIR, filename), dpi=150)
    plt.show()
    plt.close(figure)
    display(Markdown(case["reason"] + "\n\n" + case["record"]))

    record["살펴볼 이유"] = case["reason"]
    record["관련 기록"] = case["record"]
    window_records.append(record)
    rows = output.loc[selected, ["ROW_ID", "LOT_NO", "MEAS_DT", "LABEL", "EVIDENCE_SENSOR"]].copy()
    rows.insert(0, "검토번호", number)
    window_row_tables.append(rows)

window_summary = pd.DataFrame(window_records)
window_rows = pd.concat(window_row_tables, ignore_index=True)


**추가로 구분해서 볼 내용**

- 091의 `TC-Z7`, `TC-Z8`은 같은 SPHC보다 관측 전체의 온도 수준이 높음. LOT 자체 평균을 쓰면 이 차이가 기준에 들어가므로 ±3σ만으로는 잘 드러나지 않음. 전체 LOT를 임의로 1로 바꾸지는 않음.
- 058의 정기수리 정지, 061의 재가동 승온은 팀이 확정한 공정 상태임. 기존 `EXCLUDED`와 빈 라벨을 유지함.
- 정비 기록은 감지 시각과 작업 시각을 구분해 읽음. 가까운 정비가 있어야만 1이 되는 조건은 사용하지 않음.

참조: 온도 데이터사전, 조업이벤트, 정비이력, 부품교체이력, 같은 강종 비교 그래프. ±3σ는 공정 허용값이 아니라 이번 후보 선택 기준임.

## 8. 중앙값에서 얼마나, 얼마나 오래 벗어났는지 계산하기

각 LOT의 **관측 시간에서 앞뒤 20%를 자른 가운데 60%**의 센서별 중앙값을 기준으로 사용함.
양 끝 시각은 포함하고, 기존 제외 행과 해당 센서의 결측만 계산에서 뺌. `LABEL=0`만 골라 계산하지 않음.
이 중앙값은 비교용 온도이며 실제 설정온도나 강판 온도가 아님.

| TC-Z3에 붙이는 컬럼 | 뜻 |
| --- | --- |
| `TC-Z3_중앙값대비온도차` | 현재 온도 − 중앙 60%의 중앙값, ℃ |
| `TC-Z3_5도초과상승_지속초` | 5℃ 넘게 높은 상태가 현재까지 이어진 초 |
| `TC-Z3_5도초과하강_지속초` | 5℃ 넘게 낮은 상태가 현재까지 이어진 초 |
| `TC-Z3_최근5분_5도초과누적` | 최근 300초 동안 5℃를 넘긴 정도 × 시간, ℃·초 |
| `TC-Z3_중앙값대비_표준화편차` | 중앙값과의 차이 ÷ 중앙 60%의 표준편차. 평균 기준 Z-score와 구분함 |

같은 5개 컬럼을 TC-Z4~Z6에도 붙임. **5℃와 5분은 실험 기준이며 불량 판정 기준이 아님.**
기준이 800℃일 때 808℃가 10초 이어지면 초과 누적은 `(8−5)×10=30℃·초`. 강판에 전달된 열량을 뜻하지 않음.

첫 초과 관측은 0초로 시작하고 실제 시각 차이를 더함. 결측, 기존 제외 행, LOT 변경, 15초보다 긴 공백에서는 다시 시작함.
측정 사이에는 직전 값을 유지했다고 가정함. 5분이 확보되지 않으면 누적값은 비우고, 확보된 초를 별도 컬럼에 남김.

**LOT가 끝난 뒤에야 중앙값을 확정할 수 있으므로 완료된 LOT를 분석하는 컬럼임.**
기존 67개 입력은 유지하고, 새 20개를 더한 87개 목록을 `retrospective_input_columns`에 따로 저장함. 실시간 예측용 입력에 자동 추가하지 않음.

In [ ]:
THERMAL_SENSORS = ["TC-Z3", "TC-Z4", "TC-Z5", "TC-Z6"]
CENTRAL_START_FRACTION = 0.2
CENTRAL_END_FRACTION = 0.8
TEMP_LIMIT_C = 5.0
ACCUMULATION_SECONDS = 300
MAX_GAP_SECONDS = 15
thermal_suffixes = [
    "_중앙값대비온도차",
    "_5도초과상승_지속초",
    "_5도초과하강_지속초",
    "_최근5분_5도초과누적",
    "_중앙값대비_표준화편차",
]
thermal_input_columns = []
coverage_columns = []
for sensor in THERMAL_SENSORS:
    thermal_input_columns += [sensor + suffix for suffix in thermal_suffixes]
    coverage_columns.append(sensor + "_최근5분_확보초")

thermal_features = pd.DataFrame(np.nan, index=base.index, columns=thermal_input_columns + coverage_columns)
thermal_reference_records = []
thermal_summary_records = []

for lot, lot_rows in base.groupby("LOT_NO", sort=False):
    index = times.loc[lot_rows.index].sort_values().index
    clock = times.loc[index]
    start, end = clock.iloc[0], clock.iloc[-1]
    ref_start = start + (end - start) * CENTRAL_START_FRACTION
    ref_end = start + (end - start) * CENTRAL_END_FRACTION
    central = clock.ge(ref_start) & clock.le(ref_end) & not_excluded.loc[index]
    step_seconds = clock.diff().dt.total_seconds()

    for sensor in THERMAL_SENSORS:
        reference_values = values.loc[index[central], sensor].dropna()
        center = reference_values.median()
        spread = reference_values.std(ddof=0)
        thermal_reference_records.append({
            "LOT_NO": lot, "SENSOR": sensor, "LOT_START": start, "LOT_END": end,
            "REFERENCE_START_INCLUDED": ref_start, "REFERENCE_END_INCLUDED": ref_end,
            "COUNT": len(reference_values), "MEDIAN_C": center, "STD_DDOF0_C": spread,
        })
        temperature = values.loc[index, sensor]
        good = not_excluded.loc[index] & temperature.notna() & np.isfinite(center)
        deviation = temperature.subtract(center).where(good)
        connected = good & good.shift(fill_value=False) & step_seconds.gt(0) & step_seconds.le(MAX_GAP_SECONDS)
        above = deviation.gt(TEMP_LIMIT_C)
        below = deviation.lt(-TEMP_LIMIT_C)
        up_start = clock.where(above & ~(above.shift(fill_value=False) & connected)).ffill()
        down_start = clock.where(below & ~(below.shift(fill_value=False) & connected)).ffill()
        thermal_features.loc[index, sensor + "_중앙값대비온도차"] = deviation
        thermal_features.loc[index, sensor + "_5도초과상승_지속초"] = clock.subtract(up_start).dt.total_seconds().where(above, 0).where(good)
        thermal_features.loc[index, sensor + "_5도초과하강_지속초"] = clock.subtract(down_start).dt.total_seconds().where(below, 0).where(good)
        if spread > 0:
            thermal_features.loc[index, sensor + "_중앙값대비_표준화편차"] = deviation / spread

        segments = (~connected).cumsum()
        for segment, segment_rows in lot_rows.loc[index[good]].groupby(segments.loc[index[good]], sort=False):
            segment_index = times.loc[segment_rows.index].sort_values().index
            segment_times = times.loc[segment_index]
            seconds = segment_times.subtract(segment_times.iloc[0]).dt.total_seconds().to_numpy()
            excess = deviation.loc[segment_index].abs().subtract(TEMP_LIMIT_C).clip(lower=0).to_numpy()
            area = np.r_[0.0, np.cumsum(excess[:-1] * np.diff(seconds))]
            recent_area = area - np.interp(seconds - ACCUMULATION_SECONDS, seconds, area)
            coverage = np.minimum(seconds, ACCUMULATION_SECONDS)
            thermal_features.loc[segment_index, sensor + "_최근5분_확보초"] = coverage
            thermal_features.loc[segment_index, sensor + "_최근5분_5도초과누적"] = np.where(
                coverage >= ACCUMULATION_SECONDS, np.maximum(recent_area, 0), np.nan
            )

        thermal_summary_records.append({
            "LOT": lot, "센서": sensor, "기준 중앙값 ℃": center,
            "최대 절대 온도차 ℃": deviation.abs().max(),
            "최장 상승 지속초": thermal_features.loc[index, sensor + "_5도초과상승_지속초"].max(),
            "최장 하강 지속초": thermal_features.loc[index, sensor + "_5도초과하강_지속초"].max(),
            "최대 5분 초과누적 ℃·초": thermal_features.loc[index, sensor + "_최근5분_5도초과누적"].max(),
        })

thermal_reference = pd.DataFrame(thermal_reference_records)
thermal_summary = pd.DataFrame(thermal_summary_records)
thermal_complete = np.isfinite(thermal_features[thermal_input_columns]).all(axis=1)
thermal_features["THERMAL_FEATURES_COMPLETE"] = thermal_complete.map({True: "True", False: "False"})
thermal_features["RETROSPECTIVE_MODEL_ELIGIBLE"] = (thermal_complete & output["MODEL_ELIGIBLE"].eq("True")).map({True: "True", False: "False"})
output = pd.concat([output, thermal_features], axis=1)
retrospective_input_columns = input_columns + thermal_input_columns
display(thermal_reference)
display(thermal_summary)
print("기존 입력:", len(input_columns), "/ 완료 LOT 분석용 입력:", len(retrospective_input_columns))
print("새 특징 20개 계산 가능:", int(thermal_complete.sum()))
print("기존 입력과 새 특징 모두 사용 가능:", int(output["RETROSPECTIVE_MODEL_ELIGIBLE"].eq("True").sum()))


## 9. 노란 배경에서 온도와 지속 시간을 함께 보기

노란 배경은 표의 시간 범위임. 위에서부터 원래 온도, 5℃ 초과 지속 시간, 최근 5분 초과 누적을 같은 시간축으로 읽음.
같은 온도 변화라도 짧게 끝난 경우와 오래 이어진 경우를 구분하기 위한 값임. 아래 그림을 근거로 기존 라벨을 바꾸지는 않음.

In [ ]:
thermal_plot_cases = [
    ["240121", "TC-Z3", "2024-11-24 16:25", "2024-11-24 17:10"],
    ["240108", "TC-Z4", "2024-10-18 05:50", "2024-10-18 06:20"],
]
for lot, sensor, start_text, end_text in thermal_plot_cases:
    start, end = pd.Timestamp(start_text), pd.Timestamp(end_text)
    selected = base["LOT_NO"].eq(lot) & times.ge(start) & times.le(end)
    index = base.index[selected]
    ref = thermal_reference.loc[thermal_reference["LOT_NO"].eq(lot) & thermal_reference["SENSOR"].eq(sensor)].iloc[0]
    center = ref["MEDIAN_C"]
    up_col = sensor + "_5도초과상승_지속초"
    down_col = sensor + "_5도초과하강_지속초"
    area_col = sensor + "_최근5분_5도초과누적"
    display(pd.DataFrame([{
        "LOT": lot, "센서": sensor, "시작": start, "끝": end,
        "기준 중앙값 ℃": center, "최장 상승 초": output.loc[index, up_col].max(),
        "최장 하강 초": output.loc[index, down_col].max(), "최대 초과누적 ℃·초": output.loc[index, area_col].max(),
        "LABEL 1": int(output.loc[index, "LABEL"].eq("1").sum()),
        "LABEL 0": int(output.loc[index, "LABEL"].eq("0").sum()),
    }]))
    figure, axes = plt.subplots(3, 1, figsize=(13, 9), sharex=True, layout="constrained")
    for axis in axes:
        axis.axvspan(start, end, color="#f9db73", alpha=0.22)
        axis.grid(alpha=0.15)
    axes[0].plot(times.loc[index], values.loc[index, sensor], color="#355f82", label=sensor)
    axes[0].axhline(center, color="#4c8c6b", label="중앙 60%의 중앙값")
    axes[0].axhline(center + TEMP_LIMIT_C, color="#747d87", linestyle="--", label="중앙값 ±5℃")
    axes[0].axhline(center - TEMP_LIMIT_C, color="#747d87", linestyle="--")
    axes[0].set_ylabel("노내 센서 온도 (℃)")
    axes[1].plot(times.loc[index], output.loc[index, up_col], color="#df650b", label="5℃ 초과 상승")
    axes[1].plot(times.loc[index], output.loc[index, down_col], color="#337ab7", label="5℃ 초과 하강")
    axes[1].set_ylabel("현재까지 이어진 초")
    axes[1].set_ylim(0, max(1, output.loc[index, [up_col, down_col]].max().max() * 1.08))
    axes[2].plot(times.loc[index], output.loc[index, area_col], color="#854d9c", label="최근 5분의 5℃ 초과 누적")
    axes[2].set_ylabel("온도 초과량 × 시간\n(℃·초)")
    axes[2].set_ylim(0, max(1, output.loc[index, area_col].max() * 1.08))
    if output.loc[index, [up_col, down_col]].max().max() == 0:
        axes[1].text(0.5, 0.5, "5℃ 이내라 초과 지속 시간은 0초", transform=axes[1].transAxes, ha="center", color="#355f82")
    if output.loc[index, area_col].max() == 0:
        axes[2].text(0.5, 0.5, "5℃ 이내라 초과 누적값은 0", transform=axes[2].transAxes, ha="center", color="#355f82")
    for axis in axes:
        axis.legend(loc="upper left", fontsize=8)
    axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
    axes[-1].set_xlabel("2024년 실제 날짜와 시각")
    figure.suptitle(lot + " " + sensor + " | 노란 배경에서 온도와 지속 시간 함께 보기", fontsize=13)
    if SAVE_FILES:
        figure.savefig(os.path.join(FIG_DIR, "thermal_" + lot + "_" + sensor + ".png"), dpi=150)
    plt.show()
    plt.close(figure)
    if sensor == "TC-Z4":
        display(Markdown("108의 이 구간은 중앙값에서 5℃까지 벗어나지 않아 지속 시간과 누적값이 0임. 기존 ±3σ 후보 기준과 서로 다른 값을 보는 것임."))

    if lot == "240121":
        display(Markdown("앞에서 공통 움직임으로 후보에서 뺀 구간에도 5℃ 초과가 이어지는 모습은 남음. 온도 변화의 크기와 시간은 파생값으로 보관하고, 기존 라벨은 그대로 둠."))


## 10. 새 CSV와 생성 기준 저장

`LABEL`은 비교할 라벨 y. 모델 입력 X는 `dataset.json`의 `input_columns`에서만 고름.
기존 원본, 제외 구간, 입력값 67개를 유지하고 샘플 추출이나 모델 학습은 하지 않음.
새 온도 특징의 기준값과 계산 방법도 함께 저장함. 기존 67개 입력과 완료 LOT 분석용 87개 입력을 구분함.

In [ ]:
legacy_context_columns = [
    "ANY_SENSOR_CANDIDATE", "CONTROL_RESPONSE_CONTEXT", "PERIODIC_CONTEXT", "LOT_BOUNDARY_CONTEXT",
    "MAINT_WORK_OVERLAP", "SCHEDULED_REPAIR_CONTEXT", "REPAIR_PREPOST_24H_CONTEXT", "GRADE_CHANGE_NEARBY_30MIN",
    "CP_RETURN_TO_BASELINE_CONTEXT", "LOT_CONTEXT_ID", "SENSOR_CANDIDATE_IDS",
]
source_sha256 = {}
for path in [INPUT_CSV, INPUT_METADATA]:
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        while True:
            chunk = file.read(1024 * 1024)
            if not chunk:
                break
            digest.update(chunk)
    source_sha256[os.path.relpath(path, ROOT).replace("\\", "/")] = digest.hexdigest()

manifest = {
    "dataset_version": "v2", "master_file": os.path.basename(OUTPUT_CSV),
    "row_key": ["ROW_ID", "LOT_NO", "MEAS_DT"], "rows": len(output), "columns": len(output.columns),
    "label_0": int(output["LABEL"].eq("0").sum()), "label_1": int(output["LABEL"].eq("1").sum()),
    "label_na": int(output["LABEL"].eq("").sum()), "excluded_rows": int((~not_excluded).sum()),
    "score_unavailable_rows": int((not_excluded & ~valid).sum()),
    "model_eligible_rows": int(output["MODEL_ELIGIBLE"].eq("True").sum()),
    "input_columns": input_columns,
    "retrospective_input_columns": retrospective_input_columns,
    "retrospective_model_input_forbidden": [column for column in output.columns if column not in retrospective_input_columns],
    "thermal_feature_policy": {
        "sensors": THERMAL_SENSORS, "input_columns": thermal_input_columns,
        "reference": "LOT의 첫~마지막 실제 시각 기준 중앙 60%, 양끝 포함. EXCLUDED=True와 해당 센서 결측 제외, LABEL 조건 없음.",
        "center": "중앙값", "standardization": "(현재 온도 - 중앙값) / 중앙 60% 표준편차(ddof=0). 평균 기준 Z-score가 아님.",
        "central_fractions": [CENTRAL_START_FRACTION, CENTRAL_END_FRACTION],
        "deviation_threshold_c": TEMP_LIMIT_C, "rolling_seconds": ACCUMULATION_SECONDS,
        "max_observation_gap_seconds": MAX_GAP_SECONDS,
        "duration": "같은 방향으로 5℃ 초과가 처음 관측된 시점부터 현재까지의 초. 복귀/결측/제외/LOT 변경/15초 초과 공백에서 초기화.",
        "accumulation": "직전 관측의 max(abs(온도-중앙값)-5, 0)를 다음 관측까지 유지한 면적. 최근300초 경계를 걸친 부분만 잘라 합산.",
        "coverage_columns": coverage_columns,
        "incomplete_window": "연속 유효 관측이 300초 미만이면 누적값 NaN. 확보한 초는 별도 보관.",
        "reference_uses_later_observations": True,
        "allowed_use": "완료된 LOT의 사후 분석. 실시간 또는 미래 LOT 예측용 입력으로 사용하지 않음.",
        "retrospective_input_meaning": "기존 67개 + 새 온도 특징 20개를 합친 전체 입력 목록.",
        "complete_rows": int(thermal_complete.sum()),
        "retrospective_eligible_rows": int(output["RETROSPECTIVE_MODEL_ELIGIBLE"].eq("True").sum()),
        "labels_changed_by_thermal_features": False,
        "quality_limit_claimed": False,
    },
    "eligible_label_counts": output.loc[output["MODEL_ELIGIBLE"].eq("True"), "LABEL"].value_counts().to_dict(),
    "review_scope": "14 LOT 통계 계산. 기록 끝에 가까운 13회차의 Z3, Z6 공통 움직임과 주요 8개 시간 범위 검토. 전체 통계 후보 수동 검토 완료를 뜻하지 않음.",
    "sampling_policy": "샘플 추출 없음",
    "model_input_forbidden": [column for column in output.columns if column not in input_columns],
    "legacy_context_columns": legacy_context_columns,
    "legacy_context_meaning": "v1 분석 정보를 그대로 보존했으며 새 통계 라벨의 근거나 모델 입력으로 사용하지 않음.",
    "label_policy": {
        "version": LABEL_VERSION, "sensors": SENSORS, "std_ddof": 0, "threshold": 3, "operator": ">",
        "reference": "같은 LOT의 EXCLUDED=False 관측. 기존 라벨과 관계없이 센서별 유효값으로 평균과 표준편차 계산.",
        "one_definition": "21개 점수가 모두 계산되고, 검토한 공통 움직임을 뺀 뒤 절대 Z-score 3 초과 센서가 하나 이상 남은 이상 후보.",
        "zero_definition": "21개 점수가 모두 계산되고 3σ 초과가 없거나, 초과값이 검토한 공통 움직임에만 해당한 행. 정상 확정 아님.",
        "blank_definition": "기존 제외 행 또는 21개 점수를 모두 계산할 수 없는 행.",
        "domain_override": bool(review_applied), "future_prediction_evaluation": False,
        "common_review_source_sha256": REVIEWED_SOURCE_SHA256,
        "common_review_applied": bool(review_applied),
        "common_review_ranges": len(common_ranges),
        "common_review_sensors": ["TC-Z3", "OP-Z3", "TC-Z6", "OP-Z6"],
        "removed_sensor_values": int(removed_flags.to_numpy().sum()),
        "statistical_positive_rows": int(stat_labels.eq("1").sum()),
        "common_rows_to_zero": int((common_rows & output["LABEL"].eq("0")).sum()),
        "common_rows_with_other_candidates": int((common_rows & output["LABEL"].eq("1")).sum()),
        "common_review_rule": "원자료 SHA가 일치할 때 검토표의 센서별 초과 구간만 해제. 기록 끝 2시간 전체나 다른 센서를 해제하지 않음.",
    },
    "source_sha256": source_sha256,
    "split_status": "not_assigned", "sampling_applied": False, "models_trained": False,
    "thermal_domain_sources": [
        {"url": "https://www.andritz.com/products-en/environmental-solutions/metals/digitalization/advanced-furnace-control", "use": "강판의 목표 온도 경로와 노내 열관성 설명. ±5℃ 또는 5분 한계를 증명하는 자료로 사용하지 않음."},
        {"url": "https://www.itl.nist.gov/div898/handbook/pmc/section3/pmc32.htm", "use": "통계 관리한계와 품질 규격의 구분."},
    ],
}
manifest["data_kind"] = "raw"
manifest["groups"] = GROUPS
manifest["raw_sensor_columns"] = SENSORS
manifest["label_separation"] = {
    "STAT_LABEL": "공통 움직임 해제 전 같은 LOT의 절대 Z > 3",
    "DOMAIN_LABEL": "검토한 통계 후보를 모두 해제한 구역만 0. 다른 곳은 빈칸.",
    "LABEL": "기존 v2 행 라벨 유지. 구역별로도 같은 규칙을 계산함.",
}
if SAVE_FILES:
    os.makedirs(REVIEW_DIR, exist_ok=True)
    os.makedirs(META_DIR, exist_ok=True)
    output.to_csv(OUTPUT_CSV, index=False, encoding=ENC)
    limits.to_csv(os.path.join(REVIEW_DIR, "sigma_limits.csv"), index=False, encoding=ENC)
    window_summary.to_csv(os.path.join(REVIEW_DIR, "domain_windows.csv"), index=False, encoding=ENC)
    window_rows.to_csv(os.path.join(REVIEW_DIR, "domain_window_rows.csv"), index=False, encoding=ENC)
    lot_label_counts.to_csv(os.path.join(REVIEW_DIR, "lot_label_counts.csv"), encoding=ENC)
    changes.to_csv(os.path.join(REVIEW_DIR, "label_changes.csv"), index=False, encoding=ENC)
    common_ranges.to_csv(os.path.join(REVIEW_DIR, "common_pattern_ranges.csv"), index=False, encoding=ENC)
    common_audit.to_csv(os.path.join(REVIEW_DIR, "common_pattern_rows.csv"), index=False, encoding=ENC)
    common_lot_counts.to_csv(os.path.join(REVIEW_DIR, "common_pattern_lot_counts.csv"), encoding=ENC)
    thermal_reference.to_csv(os.path.join(REVIEW_DIR, "thermal_reference.csv"), index=False, encoding=ENC)
    thermal_summary.to_csv(os.path.join(REVIEW_DIR, "thermal_summary.csv"), index=False, encoding=ENC)
    with open(OUTPUT_CSV, "rb") as file:
        manifest["master_sha256"] = hashlib.file_digest(file, "sha256").hexdigest()
    with open(os.path.join(META_DIR, "dataset.json"), "w", encoding="utf-8") as file:
        json.dump(manifest, file, ensure_ascii=False, indent=2)
    print("저장 완료:", OUTPUT_CSV)
else:
    print("초안 확인: 파일을 저장하지 않음.")